In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

# Error Handling

The SDK raises typed exceptions for every HTTP error.  
This notebook shows how to:

1. Catch specific exception classes
2. Inspect error details (status code, message, headers)
3. Configure automatic retries
4. Handle connection and timeout errors
5. Log errors with structured context

## Exception hierarchy

```
InteractlyError
└── APIError (has .status_code, .message, .body, .request, .response)
    ├── AuthenticationError      (401)
    ├── PermissionDeniedError    (403)
    ├── NotFoundError            (404)
    ├── ConflictError            (409)
    ├── UnprocessableEntityError (422)
    ├── RateLimitError           (429)
    └── InternalServerError      (5xx)
APIConnectionError   (network-level, no HTTP response)
APITimeoutError      (request timed out before server responded)
```

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
from interactly import (
    AsyncWorkflowClient,
    InteractlyError,
    APIError,
    AuthenticationError,
    PermissionDeniedError,
    NotFoundError,
    ConflictError,
    RateLimitError,
    InternalServerError,
    APIConnectionError,
    APITimeoutError,
)

client = AsyncWorkflowClient()

## 1. Catching a specific exception

In [ ]:
# A well-formed ObjectId that does not exist → 404 NotFoundError.
# (A malformed id like "not-an-id" would instead raise UnprocessableEntityError/422.)
try:
    await client.workflows.get("000000000000000000000000")
except NotFoundError as e:
    print(f"404 Not Found")
    print(f"  status_code : {e.status_code}")
    print(f"  message     : {e.message}")
    print(f"  body        : {e.body}")


## 2. Catching all API errors with a single handler

In [ ]:
# Demonstrate the three branches with a single handler. We create one real workflow so the
# "found" path actually fires, then look up a mix of ids:
#   - the real id                 -> 200  -> OK
#   - a valid but unused ObjectId -> 404  -> NotFoundError  (MISS)
#   - a malformed id              -> 422  -> UnprocessableEntityError, caught by APIError (ERR)
from interactly.configs import (
    SayStaticMessageNodeConfig,
    StaticMessagesConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
)

demo = await client.workflows.create_from_config(
    WorkflowConfigFullyHydrated(
        workflow_config=WorkflowConfig(name="Error Handling Demo (13_error_handling)"),
        node_configs=[
            SayStaticMessageNodeConfig(
                name="Greeting",
                is_start=True,
                static_messages_config=StaticMessagesConfig(static_messages=["Hi!"]),
            )
        ],
        edge_configs=[],
    )
)

WORKFLOW_IDS = [
    str(demo.id),                 # exists                       -> OK
    "000000000000000000000000",   # valid ObjectId, no such wf   -> 404 NotFoundError (MISS)
    "not-an-id",                  # malformed id                 -> 422, caught below (ERR)
]

for wf_id in WORKFLOW_IDS:
    try:
        wf = await client.workflows.get(wf_id)
        print(f"  OK   {wf_id}  name={wf.name!r}")
    except NotFoundError:
        print(f"  MISS {wf_id}  (not found — skipping)")
    except APIError as e:
        print(f"  ERR  {wf_id}  HTTP {e.status_code}")

# Clean up the workflow we created for the demo.
await client.workflows.delete(demo.id)

## 3. Network and timeout errors

In [ ]:
from interactly import AsyncWorkflowClient

# Deliberately use an unreachable host to trigger a connection error
bad_client = AsyncWorkflowClient(
    base_url="http://127.0.0.1:19999",   # nothing listening here
    timeout=2.0,
    max_retries=0,                        # disable retries for this demo
)

try:
    await bad_client.workflows.list()
except APITimeoutError as e:
    print("Request timed out:", e)
except APIConnectionError as e:
    print("Connection error:", e)
except InteractlyError as e:
    print("Some other SDK error:", e)

## 4. Configuring retries

The SDK retries on 429 (rate limit), 5xx errors, and transient network failures.
By default it retries **2 times** with exponential back-off + jitter.

In [ ]:
# Increase to 4 retries for a batch import script that may hit rate limits
resilient_client = AsyncWorkflowClient(max_retries=4)

# Disable retries entirely (fail fast)
fast_fail_client = AsyncWorkflowClient(max_retries=0)

# The retry count is set per client instance at construction time.
one_retry_client = AsyncWorkflowClient(max_retries=1)
try:
    wf = await one_retry_client.workflows.get("000000000000000000000000")
except NotFoundError:
    print("Not found (after up to 1 retry)")


## 5. Authentication errors

In [ ]:
bad_auth_client = AsyncWorkflowClient(
    api_key="invalid-key",
    team_id="invalid-team",
    user_id="invalid-user",
    max_retries=0,
)

try:
    await bad_auth_client.workflows.list()
except AuthenticationError as e:
    print(f"Authentication failed (HTTP {e.status_code}): {e.message}")
except APIError as e:
    print(f"API error {e.status_code}: {e.message}")

## 6. Structured error logging

In production code, log the full error context so you can trace failures.

In [ ]:
import logging

logger = logging.getLogger("interactly_demo")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")

async def safe_get_workflow(workflow_id: str):
    try:
        return await client.workflows.get(workflow_id)
    except NotFoundError:
        logger.warning("Workflow not found", extra={"workflow_id": workflow_id})
        return None
    except AuthenticationError:
        logger.error("Auth failure — check INTERACTLY_API_KEY")
        raise  # re-raise: this is a configuration error, not a transient one
    except RateLimitError as e:
        logger.warning("Rate limited, will retry", extra={"retry_after": e.response.headers.get("retry-after")})
        raise
    except InternalServerError as e:
        logger.error(
            "Backend 5xx error",
            extra={"status_code": e.status_code, "body": str(e.body)[:200]},
        )
        raise
    except APIError as e:
        logger.error("Unexpected API error", extra={"status_code": e.status_code, "message": e.message})
        raise

result = await safe_get_workflow("000000000000000000000000")
print("Result:", result)


## 7. Accessing raw response headers

Sometimes you need to inspect headers (e.g., `X-Request-Id` for support tickets).

In [ ]:
# The error object exposes the raw httpx.Response
try:
    await client.workflows.get("000000000000000000000000")
except APIError as e:
    print("Status :", e.status_code)
    print("Headers:", dict(e.response.headers) if e.response else "no response")
